# planB — train every arm on GPU, 30 epochs, then run the counterfactual

Trains eight configurations, evaluates on the real test set, runs the
planA-protocol counterfactual, and downloads everything as one zip.

| arm | backbone | head | gap |
|---|---|---|---|
| `unconstrained` | BiLSTM | none | 3 h |
| `rayen` | BiLSTM | anchor + ray | 3 h |
| `hardnet` | BiLSTM | exact projection | 3 h |
| `brits` | BRITS | exact projection | 3 h |
| `brits_rayen` | BRITS | anchor + ray | 3 h |
| `saits` | SAITS | exact projection | 3 h |
| `blackout` | BiLSTM | exact projection | **whole day (288 steps)** |
| `hardnet --loss …` | BiLSTM | exact projection | 3 h, alternate loss weighting |

Shared: hidden 192, one MSE over 8 z-scored channels, ReLU in z-space for
curtailment, split output heads, no curtailment credit, p99.9 ramp envelope,
interpolation-residual parameterisation.

**Two fixes since the last local run, both of which will move the numbers.** The
`net_demand` feature is now supply-side, matching what the head balances to —
previously they differed by up to 4,974 MW. And the interpolation residual is
restored: planB had been predicting the dispatch level directly, forcing every
arm to learn a whole trajectory instead of a correction. Smoke test on `hardnet`:
val 3.45 → 1.57 from the residual alone.

In [ ]:
REPO  = "github.com/nm-quan/energy_modelling.git"
TOKEN = ""

import os, subprocess
os.chdir("/content")
url = f"https://{TOKEN + '@' if TOKEN else ''}{REPO}"
if not os.path.exists("/content/energy_modelling"):
    subprocess.run(["git", "clone", "-q", url], check=True)
os.chdir("/content/energy_modelling")
subprocess.run(["git", "pull", "-q"])

import torch, numpy as np
name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
mem  = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
print(f"torch {torch.__version__} | {name} | {mem:.0f} GB")

if mem > 60:   BATCH, FLAGS = 1024, "--compile"
elif mem > 35: BATCH, FLAGS = 768,  "--compile"
elif mem > 20: BATCH, FLAGS = 512,  ""
elif mem > 0:  BATCH, FLAGS = 256,  ""
else:          BATCH, FLAGS = 128,  ""

OPTS = f"--batch {BATCH} {FLAGS}".strip()
print(f"batch={BATCH}  flags={FLAGS or 'none'}")

z = np.load("data/preprocessed/hist/5min/net_dispatch_ren/prepared.npz")
print("flats:", z["Xtr_flat"].shape, "| features:", len(z["feat_cols"]))

## Constraint envelope and head self-tests

p99.9 ramps, empirical-max caps. Confirms 0/400 windows lose bridge feasibility
and both heads are exact, before any time is spent training.

In [ ]:
!python3 planB/limits.py
!python3 planB/heads.py

## Training

One cell per arm so a dropped session only loses the arm in flight. Early
stopping is `--patience 10` on validation loss; the saved checkpoint is the
**best** epoch, and history is written every epoch.

### GPU notes

- **Batch size is the lever.** Both heads walk the gap steps in a Python loop
  *once per batch*, so batch 1024 does the same 36 iterations as batch 128 for 8x
  the windows. The whole 40k window set (~450 MB) is moved onto the GPU once,
  which removes a host-to-device copy per batch.
- **`--compile` wraps the backbone only.** The heads carry data-dependent control
  flow — a 36-step sweep with a 40-step bisection inside — which graph-breaks or
  unrolls enormously. Expect one slow first batch. Helps SAITS and BRITS most,
  `nn.LSTM` least.
- **TF32 on, AMP off.** TF32 is safe: the heads are elementwise, not matmul.
  `--amp` exists and confines bf16 to the backbone, but is not default — the
  point of the layer is balance exact to ~1e-12, and bf16 near it puts the
  residual in the tens of MW.
- At batch 1024 the effective step is 8x the batch-128 tuning. If the curves look
  unstable, raise `--lr` toward 2e-3.

In [ ]:
!python3 planB/fit.py --arm unconstrained --epochs 30 --patience 10 $OPTS

In [ ]:
!python3 planB/fit.py --arm rayen --epochs 30 --patience 10 $OPTS

In [ ]:
!python3 planB/fit.py --arm hardnet --epochs 30 --patience 10 $OPTS

In [ ]:
!python3 planB/fit.py --arm brits --epochs 30 --patience 10 $OPTS

In [ ]:
!python3 planB/fit.py --arm brits_rayen --epochs 30 --patience 10 $OPTS

In [ ]:
!python3 planB/fit.py --arm saits --epochs 30 --patience 10 $OPTS

In [ ]:
!python3 planB/fit.py --arm blackout --epochs 30 --patience 10 --n-train 15000 $OPTS

## Loss variants

**"Sum the per-source MSEs" is already what the default does.** Verified:

```
((pred - true) ** 2).mean()      = 1.069212
sum of the per-source MSEs       = 8.553698
ratio                            = 8.000000   (= number of channels)
```

Same objective scaled by the channel count. Adam normalises by gradient
magnitude, so the step is essentially unchanged — only `weight_decay`'s relative
strength shifts. There is no separate flag for it because it is not a separate
loss.

What *does* change the answer is the relative weighting between sources:

| `--loss` | what it does | effect |
|---|---|---|
| `mse` (default) | MSE after z-scoring each source by its own scaler | every source equal in z-units |
| `sum_mw` | per-source MSE in raw MW, summed, no normalisation | weights by scale², coal (681) swamps battery (52) ~170x |
| `wape` | per-source `sum(err)/sum(truth)` in MW, meaned | matches the reported metric; small channels not down-weighted |

`sum_mw` is the literal reading if you do not normalise first. Its values are
~1e5 because the units are MW² — only the gradient direction matters, but the
number is not comparable to the other two.

Checkpoints are tagged with the loss (`hardnet_wape.pt`), so these do not
overwrite the defaults.

In [ ]:
!python3 planB/fit.py --arm hardnet --epochs 30 --patience 10 --loss sum_mw $OPTS
!python3 planB/fit.py --arm hardnet --epochs 30 --patience 10 --loss wape $OPTS

## Evaluation and counterfactual

In [ ]:
!python3 planB/eval.py --n 400

In [ ]:
!python3 planB/counterfactual.py --days 10

In [ ]:
import json, matplotlib.pyplot as plt
from pathlib import Path
OUT = Path("planB/results")
fig, ax = plt.subplots(1, 2, figsize=(13, 4.4))
for f in sorted(OUT.glob("*_history.json")):
    if "smoke" in f.name:
        continue
    h = json.loads(f.read_text())
    ep = range(1, len(h["train"]) + 1)
    lab = f.stem.replace("_history", "")
    ax[0].plot(ep, h["train"], lw=2, label=lab)
    ax[1].plot(ep, h["val"], lw=2, label=lab)
for a, t in zip(ax, ("train", "validation")):
    a.set_title(f"{t} loss", loc="left")
    a.set_xlabel("epoch")
    a.grid(alpha=0.2)
    a.legend(fontsize=8, frameon=False)
fig.tight_layout()
fig.savefig(OUT / "loss_curves.png", dpi=140, facecolor="white")
plt.show()

In [ ]:
from IPython.display import Image, display, Markdown
from pathlib import Path
OUT = Path("planB/results")
for f in ("counterfactual_10day.png", "loss_curves.png"):
    if (OUT / f).exists():
        display(Image(str(OUT / f)))
for f in ("eval.md", "counterfactual.md"):
    if (OUT / f).exists():
        display(Markdown((OUT / f).read_text()))

## Download to your machine

Unzip into `planB/results/` locally.

In [ ]:
import shutil, os
from google.colab import files
shutil.make_archive("/content/planB_results", "zip", "planB/results")
print(f"{os.path.getsize('/content/planB_results.zip') / 1e6:.1f} MB")
files.download("/content/planB_results.zip")

### What to look for

- **Does any arm beat interpolation** (50.6 MW dispatch MAE, 43.0 curtailment)?
  Nothing has yet, and one epoch was the standing excuse. Thirty removes it.
- **Does `rayen` beat its own anchor?** Locally the anchor alone scored 63.7 MW
  against 82.8 with the network — the network was making it worse. If that still
  holds at 30 epochs, the anchor is the model.
- **`hardnet` vs `brits` vs `saits`** share the identical head and envelope, so
  differences are the backbone alone.
- **`hardnet` vs `rayen`** and **`brits` vs `brits_rayen`** are the same head swap
  on two backbones. Same sign both times means it is the head; different means an
  interaction.
- **Coal share in the counterfactual** against the real grid's 43.4%. Every arm
  missed badly in both directions at one epoch.